In [3]:
import torch

def show_spacing_torch(dtype, center, rel_step=1e-4, count=10, name=None):
    """
    Inspect representable values around `center` for a given dtype.
    - dtype: torch.float16, torch.bfloat16, or torch.float32
    - center: scalar float (e.g., 1.0, 10000.0)
    - rel_step: step size as a fraction of `center` (e.g., 1e-4 -> center*1e-4)
    - count: number of steps on each side
    """
    name = name or str(dtype).replace('torch.', '')
    step = abs(center) * rel_step if center != 0 else rel_step
    # Build a small neighborhood around center in higher precision to avoid construction bias
    xs = torch.arange(center - step * count, center + step * count + step/2, step, dtype=torch.float64)
    # Cast to target dtype, then back to float64 for safe printing/comparison
    casted = xs.to(dtype).to(torch.float64)
    uniq = torch.unique(casted)
    diffs = torch.diff(uniq)

    print(f"\n[{name}]  around {center:g}  (rel_step={rel_step:g}, abs_step≈{step:g})")
    # Show a compact list of values
    vals_str = ", ".join([f"{v:.8g}" for v in uniq.tolist()])
    print(vals_str if len(vals_str) < 500 else vals_str[:500] + " ...")
    if diffs.numel() > 0:
        print(f"→ min spacing ≈ {diffs.min().item():.6g},  max spacing ≈ {diffs.max().item():.6g},  avg spacing ≈ {diffs.mean().item():.6g}")
    else:
        print("→ (All values collapsed to a single representable value)")

# Examples: compare near small and large magnitudes
for center in [1.0, 10000.0]:
    show_spacing_torch(torch.float16, center, rel_step=1e-4, count=12, name="float16")
    show_spacing_torch(torch.bfloat16, center, rel_step=1e-4, count=12, name="bfloat16")
    show_spacing_torch(torch.float32, center, rel_step=1e-4, count=12, name="float32")

# Extra: test a very small magnitude where fp16 starts to collapse values (near its min normal ~6e-5)
for center in [1e-4, 1e-5]:
    show_spacing_torch(torch.float16, center, rel_step=1e-2, count=12, name="float16")
    show_spacing_torch(torch.bfloat16, center, rel_step=1e-2, count=12, name="bfloat16")
    show_spacing_torch(torch.float32, center, rel_step=1e-2, count=12, name="float32")


for center in [1e-12, 1e-14]:
    show_spacing_torch(torch.float16, center, rel_step=1e-2, count=12, name="float16")
    show_spacing_torch(torch.bfloat16, center, rel_step=1e-2, count=12, name="bfloat16")
    show_spacing_torch(torch.float32, center, rel_step=1e-2, count=12, name="float32")



[float16]  around 1  (rel_step=0.0001, abs_step≈0.0001)
0.99902344, 0.99951172, 1, 1.0009766
→ min spacing ≈ 0.000488281,  max spacing ≈ 0.000976562,  avg spacing ≈ 0.000651042

[bfloat16]  around 1  (rel_step=0.0001, abs_step≈0.0001)
1
→ (All values collapsed to a single representable value)

[float32]  around 1  (rel_step=0.0001, abs_step≈0.0001)
0.99879998, 0.9989, 0.99900001, 0.99910003, 0.99919999, 0.9993, 0.99940002, 0.99949998, 0.99959999, 0.99970001, 0.99980003, 0.99989998, 1, 1.0001, 1.0002, 1.0003, 1.0003999, 1.0005, 1.0006, 1.0007, 1.0008, 1.0009, 1.001, 1.0010999, 1.0012
→ min spacing ≈ 9.98974e-05,  max spacing ≈ 0.000100017,  avg spacing ≈ 9.99992e-05

[float16]  around 10000  (rel_step=0.0001, abs_step≈1)
9984, 9992, 10000, 10008, 10016
→ min spacing ≈ 8,  max spacing ≈ 8,  avg spacing ≈ 8

[bfloat16]  around 10000  (rel_step=0.0001, abs_step≈1)
9984
→ (All values collapsed to a single representable value)

[float32]  around 10000  (rel_step=0.0001, abs_step≈1)
9988, 99

In [1]:
import numpy as np

def show_spacing(dtype, center, step, count=10):
    # generate values around 'center' and cast to given dtype
    vals = np.arange(center - step * count, center + step * count, step, dtype=np.float32)
    casted = vals.astype(dtype)
    unique_vals = np.unique(casted)
    diffs = np.diff(unique_vals)
    print(f"\n[{dtype}] around {center:g}")
    for v in unique_vals:
        print(f"{v:.8g}", end=", ")
    print(f"\n→ average spacing ≈ {np.mean(diffs):.3g}")

# Compare near 1.0 and near 10000
for center in [1.0, 10000.0]:
    show_spacing(np.float16, center, step=center*1e-4)
    show_spacing(np.float32, center, step=center*1e-4)



[<class 'numpy.float16'>] around 1
0.99902344, 0.99951172, 1, 1.0009766, 
→ average spacing ≈ 0.000651

[<class 'numpy.float32'>] around 1
0.99900001, 0.99910003, 0.99920005, 0.99930006, 0.99940008, 0.9995001, 0.99960011, 0.99970013, 0.99980015, 0.99990016, 1.0000002, 1.0001001, 1.0002003, 1.0003002, 1.0004003, 1.0005002, 1.0006003, 1.0007002, 1.0008004, 1.0009003, 
→ average spacing ≈ 0.0001

[<class 'numpy.float16'>] around 10000
9992, 10000, 10008, 
→ average spacing ≈ 8

[<class 'numpy.float32'>] around 10000
9990, 9991, 9992, 9993, 9994, 9995, 9996, 9997, 9998, 9999, 10000, 10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 
→ average spacing ≈ 1
